### Évaluation des résultats entre le gold standard normalisé et les annotations Medkit normalisées.

Le script compare :
- gold_standard_norm.csv
- annot_medkit_norm.csv

Il calcule :
- une métrique globale sur les paires alignées,
- des métriques par attribut,
- et exporte un tableau de synthèse dans un fichier CSV.


In [1]:
import pandas as pd
from pathlib import Path


# Chargement des fichiers normalisés
gold = pd.read_csv("gold_standard_norm.csv", encoding="utf-8-sig")
auto = pd.read_csv("annot_medkit_norm.csv", encoding="utf-8-sig")

# Clés d'alignement entre les deux jeux de données
key_cols = ["Medicament_norm", "Position_texte"]

# Colonnes d'attributs évaluées
attr_cols = [
    "Dosage",
    "Frequence",
    "Voie_administration",
    "Date",
    "Date_relative",
    "Contexte"
]

# Fusion gauche/droite pour identifier les correspondances exactes
merged = gold.merge(
    auto,
    on=key_cols,
    how="outer",
    suffixes=("_gold", "_auto"),
    indicator=True
)


# Fonction utilitaire de calcul des métriques
def precision_recall_f1(tp, fp, fn):
    """Retourne précision, rappel et F1 à partir des comptes TP, FP et FN."""
    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    return precision, recall, f1


In [3]:
rows = []

# Métrique globale sur les lignes alignées
TP = (merged["_merge"] == "both").sum()
FP = (merged["_merge"] == "right_only").sum()
FN = (merged["_merge"] == "left_only").sum()
VN = 0

prec = TP / (TP + FP) if (TP + FP) else 0
rec = TP / (TP + FN) if (TP + FN) else 0
f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0

rows.append(["GLOBAL", "row_key", TP, FP, FN, VN, prec, rec, f1])

# Métriques par attribut
for col in attr_cols:
    tp = fp = fn = vn = 0

    for _, row in merged.iterrows():
        g_exists = row["_merge"] in ("both", "left_only")
        a_exists = row["_merge"] in ("both", "right_only")

        g_val = "" if not g_exists or pd.isna(row.get(f"{col}_gold")) else str(row.get(f"{col}_gold")).strip()
        a_val = "" if not a_exists or pd.isna(row.get(f"{col}_auto")) else str(row.get(f"{col}_auto")).strip()

        g_pos = g_val != ""
        a_pos = a_val != ""

        if g_pos and a_pos:
            if g_val == a_val:
                tp += 1
            else:
                fp += 1
                fn += 1
        elif (not g_pos) and a_pos:
            fp += 1
        elif g_pos and (not a_pos):
            fn += 1
        else:
            vn += 1

    p = tp / (tp + fp) if (tp + fp) else 0
    r = tp / (tp + fn) if (tp + fn) else 0
    f = 2 * p * r / (p + r) if (p + r) else 0

    rows.append(["ATTR", col, tp, fp, fn, vn, p, r, f])


# Tableau final
out = pd.DataFrame(
    rows,
    columns=["scope", "metric", "VP", "FP", "FN", "VN", "precision", "recall", "F1"]
)

out.to_csv("metrics_summary.csv", index=False, encoding="utf-8-sig")

print(f"{'scope':<8} {'metric':<22} {'VP':>4} {'FP':>4} {'FN':>4} {'VN':>4} {'precision':>10} {'recall':>10} {'F1':>10}")
for _, row in out.iterrows():
    print(
        f"{row['scope']:<8} "
        f"{row['metric']:<22} "
        f"{int(row['VP']):>4} "
        f"{int(row['FP']):>4} "
        f"{int(row['FN']):>4} "
        f"{int(row['VN']):>4} "
        f"{row['precision']:>10.6f} "
        f"{row['recall']:>10.6f} "
        f"{row['F1']:>10.6f}"
    )

scope    metric                   VP   FP   FN   VN  precision     recall         F1
GLOBAL   row_key                 125    0   63    0   1.000000   0.664894   0.798722
ATTR     Dosage                   69   43  101   17   0.616071   0.405882   0.489362
ATTR     Frequence                46    6  107   35   0.884615   0.300654   0.448780
ATTR     Voie_administration      41   24   30  117   0.630769   0.577465   0.602941
ATTR     Date                     26   18   40  122   0.590909   0.393939   0.472727
ATTR     Date_relative            42    9   34  108   0.823529   0.552632   0.661417
ATTR     Contexte                 36   22   66   86   0.620690   0.352941   0.450000
